In [1]:

import os
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# DAY 19: RESULTS AND SCIENTIFIC DISCUSSION
# --------------------------------------------------

# Find the latest simulation results folder
folders = glob.glob("combined_mantle_results*")

if not folders:
    raise FileNotFoundError(
        "Run the combined mantle convection model first."
    )

result_folder = max(folders, key=os.path.getmtime)

files = glob.glob(os.path.join(result_folder, "*.npz"))

if not files:
    raise FileNotFoundError(
        "No simulation files were found."
    )

output_folder = "day19_results_discussion"
os.makedirs(output_folder, exist_ok=True)

results = []

# --------------------------------------------------
# 1. LOAD AND ANALYSE SIMULATION DATA
# --------------------------------------------------

for filename in files:

    with np.load(filename) as data:

        required = ["ra", "x", "z", "T", "u", "w"]

        if not all(key in data for key in required):
            print("Skipping incomplete file:", filename)
            continue

        Ra = float(np.asarray(data["ra"]).item())

        x = data["x"].copy()
        z = data["z"].copy()
        T = data["T"].copy()
        u = data["u"].copy()
        w = data["w"].copy()

    # Check array dimensions
    if (
        T.shape != (len(z), len(x))
        or u.shape != T.shape
        or w.shape != T.shape
    ):
        print("Skipping incompatible arrays:", filename)
        continue

    if not all(
        np.all(np.isfinite(a))
        for a in [T, u, w]
    ):
        print("Skipping non-finite data:", filename)
        continue

    if len(z) < 3 or len(x) < 3:
        print("Skipping grid that is too small:", filename)
        continue

    dz = float(np.mean(np.diff(z)))

    if dz <= 0:
        print("Skipping invalid vertical grid:", filename)
        continue

    # ----------------------------------------------
    # TEMPERATURE DIAGNOSTICS
    # ----------------------------------------------

    T_mean = np.mean(T, axis=1)

    # Conductive reference: T = 1 - z
    T_conduction = 1.0 - z

    temperature_anomaly = T_mean - T_conduction

    rms_temperature_anomaly = np.sqrt(
        np.mean(temperature_anomaly**2)
    )

    Tmin = np.min(T)
    Tmax = np.max(T)

    # ----------------------------------------------
    # VELOCITY DIAGNOSTICS
    # ----------------------------------------------

    speed = np.sqrt(u**2 + w**2)

    max_speed = np.max(speed)

    rms_speed = np.sqrt(
        np.mean(speed[1:-1, 1:-1]**2)
    )

    # ----------------------------------------------
    # NUSSELT NUMBERS
    # ----------------------------------------------

    Nu_bottom = -np.mean(
        (T[1, 1:-1] - T[0, 1:-1]) / dz
    )

    Nu_top = -np.mean(
        (T[-1, 1:-1] - T[-2, 1:-1]) / dz
    )

    denominator = max(
        abs(Nu_bottom),
        abs(Nu_top),
        1e-12
    )

    flux_mismatch = (
        abs(Nu_bottom - Nu_top) / denominator
    ) * 100

    # ----------------------------------------------
    # TEMPERATURE-FLOW CORRELATION
    # ----------------------------------------------

    T_anomaly = T - np.mean(T, axis=1, keepdims=True)

    Ta = T_anomaly[1:-1, 1:-1]
    wa = w[1:-1, 1:-1]

    if np.std(Ta) > 1e-12 and np.std(wa) > 1e-12:
        correlation = np.corrcoef(
            Ta.ravel(), wa.ravel()
        )[0, 1]
    else:
        correlation = np.nan

    # ----------------------------------------------
    # SAVE SUMMARY
    # ----------------------------------------------

    results.append([
        Ra,
        max_speed,
        rms_speed,
        Nu_bottom,
        Nu_top,
        flux_mismatch,
        rms_temperature_anomaly,
        correlation,
        Tmin,
        Tmax
    ])

    print("\nRayleigh number:", Ra)
    print("Maximum speed:", max_speed)
    print("RMS interior speed:", rms_speed)
    print("Bottom Nu:", Nu_bottom)
    print("Top Nu:", Nu_top)
    print("Flux mismatch (%):", flux_mismatch)
    print("RMS temperature anomaly:", rms_temperature_anomaly)
    print("Temperature-flow correlation:", correlation)
    print("Temperature range:", Tmin, "to", Tmax)

if not results:
    raise ValueError("No valid simulation datasets were found.")

results.sort(key=lambda row: row[0])

# --------------------------------------------------
# 2. SAVE RESULTS TABLE
# --------------------------------------------------

headers = [
    "Rayleigh_number",
    "Maximum_speed",
    "RMS_interior_speed",
    "Nu_bottom",
    "Nu_top",
    "Flux_mismatch_percent",
    "RMS_temperature_anomaly",
    "Temperature_vertical_velocity_correlation",
    "Minimum_temperature",
    "Maximum_temperature"
]

csv_path = os.path.join(
    output_folder,
    "day19_results_summary.csv"
)

with open(csv_path, "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(headers)
    writer.writerows(results)

# --------------------------------------------------
# 3. PLOT VELOCITY AND HEAT TRANSFER
# --------------------------------------------------

Ra_values = [row[0] for row in results]
max_speeds = [row[1] for row in results]
Nu_bottom = [row[3] for row in results]
Nu_top = [row[4] for row in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(Ra_values, max_speeds, "o-")
axes[0].set_xlabel("Rayleigh number")
axes[0].set_ylabel("Maximum velocity magnitude")
axes[0].set_title("Flow strength versus Ra")
axes[0].grid(True, alpha=0.3)

axes[1].plot(Ra_values, Nu_bottom, "o-", label="Bottom Nu")
axes[1].plot(Ra_values, Nu_top, "s--", label="Top Nu")
axes[1].axhline(1.0, linestyle=":", label="Conduction reference")
axes[1].set_xlabel("Rayleigh number")
axes[1].set_ylabel("Nusselt number")
axes[1].set_title("Heat transfer versus Ra")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.subplots_adjust(
    left=0.08,
    right=0.97,
    bottom=0.15,
    top=0.88,
    wspace=0.30
)

fig.savefig(
    os.path.join(
        output_folder,
        "day19_summary_comparison.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close(fig)

# --------------------------------------------------
# 4. GENERATE A DISCUSSION REPORT
# --------------------------------------------------

lowest_Ra = results[0]
highest_Ra = results[-1]

report_path = os.path.join(
    output_folder,
    "day19_discussion_report.txt"
)

with open(report_path, "w", encoding="utf-8") as file:

    file.write("DAY 19: RESULTS AND DISCUSSION\n")
    file.write("=" * 45 + "\n\n")

    file.write("Project: 2-D Numerical Modelling of Thermal ")
    file.write("Convection in the Earth's Mantle\n\n")

    file.write("1. RESULTS OVERVIEW\n")
    file.write(
        f"The available simulation files contain "
        f"{len(results)} Rayleigh-number cases.\n"
    )

    file.write(
        f"The lowest tested Rayleigh number was "
        f"{lowest_Ra[0]:g}, and the highest was "
        f"{highest_Ra[0]:g}.\n\n"
    )

    file.write("2. FLOW STRENGTH\n")
    file.write(
        f"Maximum speed at the lowest Ra: "
        f"{lowest_Ra[1]:.6g}\n"
    )
    file.write(
        f"Maximum speed at the highest Ra: "
        f"{highest_Ra[1]:.6g}\n\n"
    )

    file.write("3. HEAT TRANSFER\n")
    file.write(
        f"At the lowest Ra, bottom Nu = "
        f"{lowest_Ra[3]:.6g} and top Nu = "
        f"{lowest_Ra[4]:.6g}.\n"
    )
    file.write(
        f"At the highest Ra, bottom Nu = "
        f"{highest_Ra[3]:.6g} and top Nu = "
        f"{highest_Ra[4]:.6g}.\n\n"
    )

    file.write("4. NUMERICAL CONSISTENCY\n")

    for row in results:
        file.write(
            f"Ra = {row[0]:g}: heat-flux mismatch = "
            f"{row[5]:.4f}%; temperature range = "
            f"{row[8]:.5g} to {row[9]:.5g}.\n"
        )

    file.write("\n5. INTERPRETATION AND LIMITATIONS\n")
    file.write(
        "Compare the measured flow strength and Nusselt "
        "numbers across the tested Rayleigh numbers. "
        "Do not assume that either quantity must increase "
        "monotonically; inspect the numerical results.\n\n"
    )

    file.write(
        "Differences between top and bottom heat fluxes "
        "may reflect transient heat storage, numerical "
        "errors, or incomplete convergence. A small "
        "mismatch alone does not prove energy conservation.\n\n"
    )

    file.write(
        "The temperature-flow correlation is descriptive "
        "and does not establish causation. The saved "
        "snapshots may not represent steady state.\n\n"
    )

    file.write(
        "The simplified model does not represent every "
        "feature of the Earth's mantle. Conclusions should "
        "be limited to the assumptions and numerical "
        "formulation used in this project.\n"
    )

# --------------------------------------------------
# 5. COMPLETION
# --------------------------------------------------

print("\n" + "=" * 45)
print("DAY 19 COMPLETED")
print("=" * 45)

print("Input folder:", result_folder)
print("Output folder:", output_folder)
print("CSV:", csv_path)
print("Discussion report:", report_path)
print("Comparison plot: day19_summary_comparison.png")



Rayleigh number: 1000.0
Maximum speed: 261.5621970066285
RMS interior speed: 93.76076219471409
Bottom Nu: 5.836153095734757
Top Nu: 7.763581696574158
Flux mismatch (%): 24.826538525252055
RMS temperature anomaly: 0.2518384667502802
Temperature-flow correlation: 0.9820315094148379
Temperature range: 0.0 to 1.0

Rayleigh number: 2500.0
Maximum speed: 564.3629566969514
RMS interior speed: 183.25955899944853
Bottom Nu: 9.746131680905368
Top Nu: 12.454051976431867
Flux mismatch (%): 21.743287250213715
RMS temperature anomaly: 0.23461324072999876
Temperature-flow correlation: 0.9229786359791136
Temperature range: 0.0 to 1.0

Rayleigh number: 500.0
Maximum speed: 133.09757056314587
RMS interior speed: 46.9836503556294
Bottom Nu: 7.0535586228536165
Top Nu: 7.6511121496079335
Flux mismatch (%): 7.8100217990522784
RMS temperature anomaly: 0.2892066986086916
Temperature-flow correlation: 0.972090545507238
Temperature range: 0.0 to 1.0

Rayleigh number: 5000.0
Maximum speed: 1064.8532313417584
RM